# Generative Adversarial Networks (GANs)

**Vanilla GAN - Simple MLP GAN - "Hello GAN"**

A GAN consists of two neural networks competing in a game:
1. **The Generator ($G$):** A "counterfeiter" trying to create realistic images from random noise.
2. **The Discriminator ($D$):** A "detective" trying to distinguish between real data and the counterfeits.

### Learning Objectives:
1. Understand the **Adversarial Objective** (Minimax game).
2. Implement a simple GAN using Fully-Connected layers.
3. Target specific FashionMNIST classes for faster convergence.
4. Visualize how the Generator improves over training epochs.

## 1. Theory: The Minimax Game

The training of a GAN is formulated as a zero-sum game with the following objective function:

$$\min_G \max_D V(D, G) = \mathbb{E}_{x \sim p_{data}(x)}[\log D(x)] + \mathbb{E}_{z \sim p_z(z)}[\log(1 - D(G(z)))]$$

#### What this means:
- **Discriminator ($D$)** wants to maximize this: it wants to output $1$ for real data ($x$) and $0$ for fake data ($G(z)$).
- **Generator ($G$)** wants to minimize the second term: it wants $D(G(z))$ to be close to $1$ (tricking the detective).

> **Note:** In practice, we train $G$ by maximizing $\log(D(G(z)))$ for better gradient flow early on.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset

import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Targeted Dataset Setup
Lets focus on just **one or two classes**. 
Let's target **Sneakers** and **T-shirts**.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
    # transforms.Normalize((-1, 1), (1, 1))
])

# Load Full Dataset
full_train = torchvision.datasets.FashionMNIST(root='./data', train=True, transform=transform, download=True)

# Filter for T-shirts (0) and Sneakers (7)
target_classes = [0, 7]
indices = [i for i, label in enumerate(full_train.targets) if label in target_classes]
train_subset = Subset(full_train, indices)

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)

print(f"Total samples for classes {target_classes}: {len(train_subset)}")


## 3. Defining the Players
We'll use simple MLPs. The Generator starts with a latent vector (100-dim noise) and expands it into a 28x28 image.

In [ ]:

latent_size = 100 # Why random noise since we are generating images? Because we want to generate images from scratch!

class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.main = nn.Sequential(
            nn.Linear(latent_size, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2),
            nn.Linear(1024, 784),
            nn.Tanh() # Tanh ensures pixels are in range [-1, 1]
        )

    def forward(self, x):
        return self.main(x).view(-1, 1, 28, 28)

class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid() # Probability: 1=Real, 0=Fake
        )

    def forward(self, x):
        return self.main(x)

modelGen = Generator().to(device)
modelDis = Discriminator().to(device)


In [ ]:
modelGen

In [ ]:
modelDis

## 4. The Adversarial Training Loop
We alternate training between the detective ($D$) and the counterfeiter ($G$).

In [ ]:
criterion = nn.BCELoss() # Loss function
lr = 0.001 # Learning rate
beta1 = 0.5 # Common Beta1 for Adam in GANs

optimizerD = optim.Adam(modelDis.parameters(), lr=lr, betas=(beta1, 0.999))
optimizerG = optim.Adam(modelGen.parameters(), lr=lr, betas=(beta1, 0.999))

real_label = 1.0
fake_label = 0.0

# Fixed noise for consistent visualization of progress
fixed_noise = torch.randn(16, latent_size, device=device)


In [ ]:

epochs = 20
print("GAN training...")

for epoch in range(epochs):
    for i, (images, _) in enumerate(train_loader):
        batch_size = images.size(0) # Batch size of 32
        
        # Update Discriminator: Maximize log(D(x)) + log(1 - D(G(z)))
        modelDis.zero_grad()

        # Real images
        images = images.to(device)
        labels = torch.full((batch_size,), real_label, device=device)
        output = modelDis(images).view(-1)
        errD_real = criterion(output, labels)
        errD_real.backward()
        
        # Fake images
        noise = torch.randn(batch_size, latent_size, device=device)
        fake = modelGen(noise)
        labels.fill_(fake_label)
        output = modelDis(fake.detach()).view(-1)
        errD_fake = criterion(output, labels)
        errD_fake.backward()
        optimizerD.step()

        # Update Generator: Maximize log(D(G(z)))
        modelGen.zero_grad()

        labels.fill_(real_label) # Tricking D to think fake is real
        output = modelDis(fake).view(-1)
        errG = criterion(output, labels)
        errG.backward()
        optimizerG.step()

    print(f"[{epoch+1}/{epochs}] Loss_D: {errD_real+errD_fake:.4f} Loss_G: {errG:.4f}")
    
    # Optional: Every 5 epochs, plot generated results
    if (epoch + 1) % 5 == 0:
        with torch.no_grad():
            fake_imgs = modelGen(fixed_noise).detach().cpu()
            plt.figure(figsize=(4,4))
            for k in range(16):
                plt.subplot(4,4,k+1)
                plt.imshow(fake_imgs[k].squeeze(), cmap='gray')
                plt.axis('off')
            plt.show()

In [ ]:
batch_size = images.size(0) # Batch size
print(batch_size)

## 5. Visualizing Final Performance
Let's generate a batch of samples and see if our model learned to create realistic clothing/items.

In [ ]:

modelGen.eval()
with torch.no_grad():
    final_noise = torch.randn(64, latent_size, device=device) # 64, latent_size=100
    final_fakes = modelGen(final_noise).cpu()
    
    plt.figure(figsize=(10,10))
    for i in range(64): # 64 images of 8x8 grid
        plt.subplot(8,8,i+1) 
        plt.imshow(final_fakes[i].squeeze(), cmap='gray')
        plt.axis('off')
    plt.suptitle("Generated Items (Sneakers or T-shirts)")
    plt.show()


## 6. Key Challenges in GAN Training

Training GANs is notoriously tricky. Did you notice any of these issues?

1. **Mode Collapse:** The Generator gets stuck producing the exact same image (e.g., only one specific sneaker) because it found a way to trick $D$ with that one sample.
2. **Vanishing Gradients:** If the Discriminator is *too good*, the Generator gets no feedback on how to improve.
3. **Convergence Oscillation:** Instead of settling, the two networks might keep going in circles in the loss space.

## 7. Workshop Exercises

1. **Class Swap:** Change the `target_classes` to `[5, 9]` (Sandal and Ankle boot). How much longer does it take to get recognizable shapes?
2. **Noise Impact:** Increase the `latent_size` to 200. Does this improve the variety of generated items, or does it make training harder?
3. **Activation Battle:** Replace `LeakyReLU` with standard `ReLU`. Does the model still converge, or does it collapse?

# Finally
As developed, the model takes the Fashion MNIST dataset (28x28 grayscale images of clothing) as input.
The outcome is a Generative Adversarial Network (GAN) that generates synthetic images of clothing items.